<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Neural Networks and Deep Learning
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

Session 5 fitted logistic regression to the feature table, and your trio then built and tuned a model of its own. Today we compare the five, take logistic regression apart into a single neuron, and build networks from it. Four kinds of activity are marked with **»**:

* **» Predict** — *before* running, decide what the output will be, then run to check. Only used where you have already met the code.
* **» Investigate** — run a working cell, then answer a question about it, from the output or by changing one thing.
* **» Modify** — change a working cell so it answers the next question.
* **» Make** — write the code yourself. The pattern is always somewhere above. **Hint** blocks fold open if you need them: try first.

Part way through, an **Independent practice** slide hands over to you for a long stretch of self-paced work; a **Regroup** slide marks where we come back together.

The notebook is **one environment**: the imports, the tables and the split exist in every cell after them, so run the cells in order. After a restart, or if a cell complains that `X_train` is not defined, use **Runtime → Run before** to catch up. Every practice cell needs only the setup in section 1.

**MIMIC-IV outputs are not in the published slides or notes** beyond summary figures. The data use agreement forbids sharing the data, so you see the rest only when you run the cells yourself. For the same reason, never paste MIMIC rows into an AI assistant or share them; asking about your code is fine.

> Worked solutions, with explanations, are in the solutions notebook, released after the session.

## Welcome

**Goal:** by the end of today you will be able to say what a neural network adds to logistic regression, fit one, and judge whether it is worth having.

It starts with two talks, then the five models your groups built.

### Before we start
1. Open this notebook in Colab: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/06_Neural_Networks_and_Deep_Learning.ipynb).
2. **File → Save a copy in Drive**, and work in your copy.
3. Run the **setup** cells in section 1 now, so they are ready when the talks end.

## Intended Learning Outcomes
By the end of this session you will be able to:

1. **Explain** how a neural network builds on logistic regression, a single neuron, and how it learns by gradient descent.
2. **Distinguish** a real difference between models from the variation between splits and between random starting weights.
3. **Judge** whether a neural network is worth its cost over logistic regression for a given clinical problem.
4. **Design** a hyperparameter search for a neural network, and **test** the chosen network once on patients it has never seen.

## Session outline
1. Setup, and two talks: distance functions, and explainable AI
2. Five models on one dataset
3. From one neuron to a network
4. How a network learns
5. Deep learning
6. Independent practice: a neural network for in-hospital death
7. Closing

# 1. Setup
The same course views, table and patient split as Session 5 and your trio's task. The first time, Colab asks you to sign in: use the **Google account linked to PhysioNet**.

**Run this once.**

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_gbq
import sklearn

# scikit-learn steps return tables with their column names, not bare arrays
sklearn.set_config(transform_output='pandas')

# The Google Cloud project that pays for your queries. After the course, use your own.
PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.icustays
df_icustays = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.icustays`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_vitalsign
df_vitals = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_vitalsign`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")
# Full MIMIC-IV: physionet-data.mimiciv_3_1_derived.first_day_lab
df_labs = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.first_day_lab`")

print(len(df_patients), len(df_admissions), len(df_icustays), len(df_vitals), len(df_diagnoses), len(df_labs))

`df_stays`: one row per ICU stay, as in Session 5.

In [ ]:
df_stays = df_icustays.merge(df_admissions, on=['subject_id', 'hadm_id'])
df_stays = df_stays.merge(df_patients, on='subject_id')
df_stays = df_stays.merge(df_vitals, on=['subject_id', 'stay_id'])
df_stays = df_stays.merge(df_labs[['stay_id', 'hemoglobin_min', 'hematocrit_min']], on='stay_id', how='left')

df_stays['died'] = df_stays['hospital_expire_flag'].astype(int)
age = df_stays['anchor_age'] + df_stays['admittime'].dt.year - df_stays['anchor_year']
df_stays['age'] = age.astype(int)
time_to_death = df_stays['deathtime'] - df_stays['intime']
df_stays['dead_by_24h'] = time_to_death.dt.total_seconds() < 24 * 60 * 60

# These columns arrive as Decimal objects (BigQuery NUMERIC), which pandas cannot average: make them floats
for column in ['temperature_min', 'temperature_max', 'temperature_mean', 'hemoglobin_min', 'hematocrit_min', 'los']:
    df_stays[column] = df_stays[column].astype(float)

print(df_stays.shape)

`df_model`: the first ICU stay of each admission, with admission type and race grouped and one-hot encoded, as in Session 5.

In [ ]:
# The first ICU stay of each admission. Sorted by stay_id so the split below is the same for everyone
df_model = df_stays.sort_values(['intime', 'stay_id'])
df_model = df_model.drop_duplicates('hadm_id', keep='first')
df_model = df_model.sort_values('stay_id')

grouping = {
    'EW EMER.': 'EMERGENCY', 'DIRECT EMER.': 'EMERGENCY',
    'URGENT': 'URGENT',
    'ELECTIVE': 'ELECTIVE', 'SURGICAL SAME DAY ADMISSION': 'ELECTIVE',
    'OBSERVATION ADMIT': 'OBSERVATION', 'EU OBSERVATION': 'OBSERVATION',
    'DIRECT OBSERVATION': 'OBSERVATION', 'AMBULATORY OBSERVATION': 'OBSERVATION',
}
admission_group = df_model['admission_type'].replace(grouping)
adm_dummies = pd.get_dummies(admission_group, prefix='adm').astype(int)

race_group = df_model['race'].str.replace(r' -.*$', '', regex=True)
race_group = race_group.str.replace(' OR ', '/')
race_group = race_group.replace(['UNKNOWN', 'UNABLE TO OBTAIN', 'PATIENT DECLINED TO ANSWER', 'OTHER'], 'UNKNOWN/OTHER')
race_dummies = pd.get_dummies(race_group, prefix='race').astype(int)

df_model = pd.concat([df_model, adm_dummies, race_dummies], axis=1)
print(df_model.shape)

`df_features` and `features`: Session 5's table and its 25 model columns. Imputation and scaling learn from rows, so they come after the split.

In [ ]:
# Clinical limits are fixed numbers, not learnt from the rows, so they can come before the split
df_model['glucose_max'] = df_model['glucose_max'].mask(df_model['glucose_max'] > 2000)
df_model['heart_rate_max'] = df_model['heart_rate_max'].mask(df_model['heart_rate_max'] > 250)

temperature_gap = df_model['temperature_mean'].isna()
df_model['temperature_missing'] = temperature_gap.astype(int)

is_icd9_diabetes = df_diagnoses['icd_code'].str.startswith('250')
is_icd10_diabetes = df_diagnoses['icd_code'].str.startswith('E10') | df_diagnoses['icd_code'].str.startswith('E11')
df_diabetes = df_diagnoses[is_icd9_diabetes | is_icd10_diabetes]
has_code = df_model['hadm_id'].isin(df_diabetes['hadm_id'])
df_model['diabetes'] = has_code.astype(int)

numeric_features = ['age', 'heart_rate_mean', 'heart_rate_max', 'resp_rate_mean', 'sbp_min', 'glucose_max']
binary_features = ['temperature_missing', 'diabetes'] + list(adm_dummies.columns) + list(race_dummies.columns)
other_columns = ['subject_id', 'anchor_year_group', 'died', 'los', 'dead_by_24h', 'hemoglobin_min', 'hematocrit_min', 'temperature_mean']

# The model's columns: Session 4's, without diabetes, which is only coded after discharge
kept_binary = binary_features.copy()
kept_binary.remove('diabetes')
features = numeric_features + ['temperature_mean'] + kept_binary

df_features = df_model[other_columns + numeric_features + binary_features].copy()
df_features = df_features.dropna(subset=numeric_features)
print(df_features.shape)
print('Stays that ended in death:', round(df_features['died'].mean(), 3))

`df_train` and `df_test`: Session 5's split of the **patients**, 80% to training and 20% to test, with the labels `y_train` and `y_test`.

In [ ]:
from sklearn.model_selection import train_test_split

patients = df_features['subject_id'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=123)

in_train = df_features['subject_id'].isin(train_patients)
in_test = df_features['subject_id'].isin(test_patients)
df_train = df_features[in_train].copy()
df_test = df_features[in_test].copy()

y_train = df_train['died']
y_test = df_test['died']
print(len(df_train), 'training rows,', len(df_test), 'test rows')

`X_train` and `X_test`: the features imputed and scaled by a pipeline fitted on the training rows only, as in Session 5 section 4.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

prepare = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
prepare.fit(df_train[features])

X_train = prepare.transform(df_train[features])
X_test = prepare.transform(df_test[features])

## Talks: distance functions, and explainable AI
Five minutes from each of this week's groups.

**While they talk, write down one question you would ask them.** Afterwards, two or three of you will ask yours.

# 2. Five models on one dataset
Before today each group built, tuned and tested one type of model: the same table, the same patient split, the same rules. Each searched its hyperparameters with `GridSearchCV` on the training rows, and used the test rows once, at the end.

In [ ]:
import pandas as pd

# The group's reported results: one row per model, from each group's table
trio_results = pd.DataFrame(
    [
        ['Logistic regression (tuned)', 'C = 0.01 (L2, ridge)', 0.789, 0.016, 0.798],
        ['Decision tree', 'max_depth = 8, min_samples_leaf = 100', 0.752, 0.017, 0.756],
        ['K-nearest neighbours', 'n_neighbors = 200, weights = distance', 0.785, 0.017, 0.786],
        ['Support vector machine', 'C = 0.1, gamma = 0.01', 0.769, 0.026, 0.770],
        ['Neural network', 'hidden_layer_sizes = (50,), alpha = 1', 0.807, 0.018, 0.811],
    ],
    columns=['Model', 'Chosen', 'CV AUC', 'CV spread', 'Test AUC'],
)
trio_results.set_index('Model')

## One split, five models
Each bar is a model's cross-validated AUC on the training rows: the mean ± one standard deviation across the five folds. The diamond is its AUC on the shared test patients. Grey: Session 5's logistic regression, with the default `C = 1`, on 30 other splits of the patients, as in its section 7.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Logistic regression on 30 splits of the patients, as in Session 5 section 7
pipeline_lr = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
split_aucs = []
for seed in range(30):
    patients_a, patients_b = train_test_split(patients, test_size=0.2, random_state=seed)
    rows_a = df_features[df_features['subject_id'].isin(patients_a)]
    rows_b = df_features[df_features['subject_id'].isin(patients_b)]
    pipeline_lr.fit(rows_a[features], rows_a['died'])
    prob_b = pipeline_lr.predict_proba(rows_b[features])[:, 1]
    split_aucs.append(roc_auc_score(rows_b['died'], prob_b))

# One row per model, the 30 splits in a row of their own at the top
models = trio_results.reset_index(drop=True)
colours = ['#0000cd', '#c0392b', '#27ae60', '#8e44ad', '#d35400']
rng = np.random.default_rng(0)
height = rng.uniform(-0.15, 0.15, len(split_aucs))

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.scatter(split_aucs, len(models) + height, color='#999999', s=25, alpha=0.7)
for i, row in models.iterrows():
    y = len(models) - 1 - i
    low = row['CV AUC'] - row['CV spread']
    high = row['CV AUC'] + row['CV spread']
    ax.plot([low, high], [y, y], color=colours[i], linewidth=6, alpha=0.35, solid_capstyle='butt')
    ax.plot(row['CV AUC'], y, marker='|', color=colours[i], markersize=14, markeredgewidth=2)
    ax.plot(row['Test AUC'], y, marker='D', color=colours[i], markersize=8)
ax.set_yticks(range(len(models) + 1))
ax.set_yticklabels(list(models['Model'][::-1]) + ['Logistic regression, default C = 1 (weak L2),\n30 other splits'], fontsize=9)
ax.set_xlabel('AUC   (bar: cross-validated, mean ± 1 SD over 5 folds;  ◆: the shared test split)')
ax.set_ylim(-0.6, len(models) + 0.5)
for side in ['top', 'right', 'left']:
    ax.spines[side].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.show()

**» Investigate** — **Which differences between the models would you believe?** Compare the gaps between the diamonds with the length of the bars and the spread of the grey dots.


## Hyperparameters: what you did
*   **Hyperparameters** are chosen before fitting: `C`, the strength of the penalty in logistic regression and the SVM (smaller is stronger); a tree's depth; the number of neighbours. **Parameters** are learned by `fit`: the coefficients
*   `GridSearchCV` fits every combination by cross-validation on the training rows, and keeps the best:

```python
search = GridSearchCV(pipeline, grid, cv=GroupKFold(n_splits=5), scoring='roc_auc')
search.fit(df_train[features], df_train['died'], groups=df_train['subject_id'])
print(search.best_params_, search.best_score_)
```

*   The test rows once, at the end, on the chosen model
*   **Best value at the edge of the grid?** The search has not finished: extend it that way. **Scores closer together than the spread between folds?** The choice hardly matters. Searching `C` from 0.01 to 100 did both: the smallest won, and every value scored within 0.002 of it, against a spread between folds of 0.016

# 3. From one neuron to a network
The building block of a neural network is the **artificial neuron**. Each input is multiplied by a **weight**, the results are added up with a **bias**, and the sum goes through an **activation function**, which gives the output. The box here shows one activation function, ReLU, which comes up shortly.

![An artificial neuron](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Artifical-Neuron.svg)

**» Investigate** — You have met a single neuron already. **Which model from Session 5 is one neuron with the sigmoid as its activation, and what were its weights and bias called there?**

## Logistic regression is one neuron
With the sigmoid as its activation, the neuron's output is a probability: Session 5's logistic regression.

| Logistic regression | Neuron |
| :-- | :-- |
| coefficients, `coef_` | weights |
| intercept, `intercept_` | bias |
| sigmoid | activation function |

## Can one neuron learn this?
Patients who are cold and patients with a fever both die more often.

Each dot is a tenth of the patients with a recorded first-day mean temperature, about 800 each, and the share of them who died.

The line is a single neuron (logistic regression) fitted to temperature alone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Published, so each dot is a tenth of the patients, hundreds each, never fewer than 10
with_temperature = df_features.dropna(subset=['temperature_mean'])
tenth = pd.qcut(with_temperature['temperature_mean'], 10)
by_tenth = with_temperature.groupby(tenth, observed=True).agg(
    temperature=('temperature_mean', 'mean'), died=('died', 'mean'), patients=('died', 'size'))
by_tenth = by_tenth[by_tenth['patients'] >= 10]

temperature_train = df_train.dropna(subset=['temperature_mean'])
temperature_test = df_test.dropna(subset=['temperature_mean'])
temperatures = pd.DataFrame({'temperature_mean': np.linspace(35.5, 38.5, 200)})


def fit_temperature(model):
    """Fit a model to temperature alone; return its curve and its test AUC."""
    pipeline = make_pipeline(StandardScaler(), model)
    pipeline.fit(temperature_train[['temperature_mean']], temperature_train['died'])
    curve = pipeline.predict_proba(temperatures)[:, 1]
    prob = pipeline.predict_proba(temperature_test[['temperature_mean']])[:, 1]
    return curve, roc_auc_score(temperature_test['died'], prob)


def draw_temperature(lines):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.scatter(by_tenth['temperature'], by_tenth['died'], color='#444444', s=40, zorder=3, label='Share who died, each tenth of patients')
    for name, (curve, auc), colour in lines:
        ax.plot(temperatures['temperature_mean'], curve, color=colour, linewidth=2, label=f'{name}: test AUC {auc:.2f}')
    ax.set_xlabel('Mean temperature, first day (°C)')
    ax.set_ylabel('Risk of death')
    ax.set_ylim(0, 0.5)
    ax.legend(frameon=False, fontsize=9, loc='upper center')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()


one_neuron = fit_temperature(LogisticRegression())
draw_temperature([('Logistic regression', one_neuron, '#0000cd')])

**» Investigate** — Compare the line with the dots at each end of the temperature range. **Which patients does this model get most wrong? Could any weight and bias put it right?**

## A hidden layer
*   Put several neurons side by side, each with its own weights: a **hidden layer**. Every feature goes to every hidden neuron
*   An **output neuron** combines their outputs, through the sigmoid, into a probability. It is a logistic regression whose features are the hidden neurons

![One input, a hidden layer of ten ReLU neurons, and a sigmoid output](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Hidden-Layer.svg)

## Why a hidden layer can bend
*   Hidden neurons usually use **ReLU**: zero below 0, a straight line above. Each hidden neuron adds one bend
*   With enough bends, the output can follow a curve that falls and then rises
*   Without an activation, a hidden layer is a weighted sum of weighted sums: still one straight line

![ReLU, sigmoid and tanh](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Activation-Functions.svg)

## The same feature, ten hidden neurons
The dots and the logistic regression from before, with a network of ten hidden neurons fitted to the same training patients.

In [ ]:
# The number of hidden neurons
hidden = 10

ten_neurons = fit_temperature(MLPClassifier(hidden_layer_sizes=(hidden,), random_state=0, max_iter=2000))
draw_temperature([('Logistic regression', one_neuron, '#0000cd'), (f'{hidden} hidden neurons', ten_neurons, '#c0392b')])

**» Investigate** — In your notebook, change `hidden = 10` in the cell above to 1, then 2, then 100. **What is the smallest number of hidden neurons that can follow the U?** What does adding more change?

## One, two, ten, a hundred hidden neurons
The same network on temperature alone, with each number of hidden neurons.

In [ ]:
sizes = [1, 2, 10, 100]

fig, axes = plt.subplots(1, len(sizes), figsize=(12, 3.2), sharey=True)
for ax, size in zip(axes, sizes):
    curve, auc = fit_temperature(MLPClassifier(hidden_layer_sizes=(size,), random_state=0, max_iter=2000))
    ax.scatter(by_tenth['temperature'], by_tenth['died'], color='#444444', s=20, zorder=3)
    ax.plot(temperatures['temperature_mean'], curve, color='#c0392b', linewidth=2)
    ax.set_title(f'{size} hidden: test AUC {auc:.2f}', fontsize=10)
    ax.set_xlabel('Mean temperature (°C)')
    ax.set_ylim(0, 0.5)
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel('Risk of death')
plt.tight_layout()
plt.show()

*   **This is not overfitting.** A hundred neurons follow the same U as two, and score the same on the test patients
*   With one feature, hundreds of patients share each temperature, some who died and some who survived, and nothing in the input tells them apart: there is nothing individual to memorise
*   So the best any model can do is the death rate at each temperature. Scoring test patients by that alone also gives 0.65: a bigger network cannot beat the information in its inputs

## Neural networks in scikit-learn
`MLPClassifier`, a **multi-layer perceptron**: the network you have just seen, used the same way as every other model.

**Back to the whole table:** all 25 features, not just temperature. `X_train` and `X_test` are the features from the setup, imputed and scaled on the training rows as in Session 5; `y_train` and `y_test` are `died`.

*   `hidden_layer_sizes=(10,)`: one hidden layer of ten neurons. `(10, 10)` would be two
*   `random_state` fixes the random starting weights: section 4 explains
*   `max_iter`: the most steps it may take while learning

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier

model_net = MLPClassifier(hidden_layer_sizes=(10,), random_state=0, max_iter=1000)
model_net.fit(X_train, y_train)

prob_net = model_net.predict_proba(X_test)[:, 1]
print('AUC, test rows:', round(roc_auc_score(y_test, prob_net), 3))

**» Investigate** — The table has 25 features, and last session's logistic regression learned 26 numbers: 25 coefficients and an intercept (which as a neuron are 25 weights and a bias). **How many weights and biases has this network learned?** Work it out from the picture of a hidden layer, then run the next cell to check.


In [ ]:
# coefs_ and intercepts_ hold one entry per layer: [0] is the hidden layer, [1] the output neuron
# A shape is (rows, columns): here, inputs by neurons
print('Hidden layer, weights:', model_net.coefs_[0].shape)
print('Hidden layer, biases: ', model_net.intercepts_[0].shape)
print('Output, weights:      ', model_net.coefs_[1].shape)
print('Output, bias:         ', model_net.intercepts_[1].shape)

total = model_net.coefs_[0].size + model_net.intercepts_[0].size + model_net.coefs_[1].size + model_net.intercepts_[1].size
print('Total:', total)

**» Predict** — scikit-learn's default network has **100** hidden neurons: 2,701 weights and biases. Logistic regression reached an AUC of 0.80 on the test rows. **Write down the test AUC you expect from the default network.**

In [ ]:
model_default = MLPClassifier(random_state=0, max_iter=1000)
model_default.fit(X_train, y_train)

prob_default = model_default.predict_proba(X_test)[:, 1]
prob_default_train = model_default.predict_proba(X_train)[:, 1]
print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_default), 3))
print('AUC, training rows:', round(roc_auc_score(y_train, prob_default_train), 3))

# 4. How a network learns
There is no formula for a network's best weights, so, like logistic regression in Session 5's appendix, it searches:

1. Start from small **random** weights: `random_state` fixes them
2. **Forward:** predict every training row, and measure the **loss**: the log loss, from Session 5's appendix
3. **Backward:** work out, from the output layer back, which way each weight should move to lower the loss. This is **backpropagation**
4. Move every weight a small step that way: **gradient descent**. The **learning rate** sets the size of the step
5. Repeat until the loss stops falling, or `max_iter` runs out

## One step of training
One training patient, forward through a network that has not learned much yet: features 2.1 and 1.0 give a predicted probability of 0.8, for a patient who survived (0). The **cross-entropy loss**, the log loss, measures the miss. Backpropagation then works out how each weight, each line here, should change to shrink it.

![One step of training](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Training-ANN-4.svg)

## Gradient descent
*   Training adds up the loss over every training row, then asks of each weight: which way lowers it? That direction is the **gradient**
*   Each step moves every weight a little way downhill. The **learning rate** sets the size of the step: too small and training crawls, too large and it overshoots
*   The animation: the search on a loss with two weights, from several starting points

<video width="640" height="360" controls loop autoplay muted>
  <source src="https://upload.wikimedia.org/wikipedia/commons/4/4c/Gradient_Descent_in_2D.webm" type="video/webm">
  Your browser does not support the video tag.
</video>

By Gpeyre, CC BY-SA 4.0, [commons.wikimedia.org/wiki/File:Gradient_Descent_in_2D.webm](https://commons.wikimedia.org/wiki/File:Gradient_Descent_in_2D.webm)

## Watching it learn
`loss_curve_` holds the loss on the training rows after each pass through them, here for the default network from the Predict.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(model_default.loss_curve_)
ax.set_xlabel('Passes through the training rows')
ax.set_ylabel('Log loss, training rows')
ax.set_ylim(bottom=0)
plt.show()

**» Investigate** — The training loss keeps falling for hundreds of passes. This cell stops the same network after `passes` passes. **Try 25, 100 and 400: after how many passes does it do best on the test patients?** What was it learning in the passes after that?

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# Stopping early warns that training has not finished: expected here
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# The same default network, stopped after this many passes
passes = 25

model_short = MLPClassifier(random_state=0, max_iter=passes)
model_short.fit(X_train, y_train)

prob_short_train = model_short.predict_proba(X_train)[:, 1]
prob_short = model_short.predict_proba(X_test)[:, 1]
print('AUC, training rows:', round(roc_auc_score(y_train, prob_short_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_short), 3))

## Too many weights
A network has its own version of logistic regression's regularisation: `alpha`, a penalty on large weights. It runs the **opposite way** to `C`: a larger `alpha` is a stronger penalty.

A large network, two hidden layers of 200 neurons, with almost no penalty:

In [ ]:
model_big = MLPClassifier(hidden_layer_sizes=(200, 200), alpha=0.00001, random_state=0, max_iter=500)
model_big.fit(X_train, y_train)

prob_big_train = model_big.predict_proba(X_train)[:, 1]
prob_big = model_big.predict_proba(X_test)[:, 1]
print('AUC, training rows:', round(roc_auc_score(y_train, prob_big_train), 3))
print('AUC, test rows:    ', round(roc_auc_score(y_test, prob_big), 3))

**» Investigate** — **Change `alpha` to 1 and run it again.** What happens to each AUC? Which of Session 4's Models A, B and C was the network before the change?

# 5. Deep learning
*   **Deep** networks have many hidden layers: tens or hundreds, not one
*   Each layer builds on the one before, so later layers can respond to more abstract patterns
*   They need large datasets, and often special arrangements of neurons for images and sequences

## Images are pixels
*   To a network, an image is a grid of numbers, one per pixel. Flattened into a row, a 28 × 28 image becomes 784 features
*   An ordinary network, like `MLPClassifier`, sees them as 784 separate columns: it is not told which pixels are next to each other

![An image flattened into a network's inputs](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Image-ANN.svg)

## What hidden neurons learn
*   A cartoon: a network telling starfish from sea urchins. Each hidden neuron responds to one feature: a star shape, spots, a ring, stripes
*   Training strengthens connections between features that occur together (left); a new image's features point to the answer (right)
*   Real hidden neurons rarely match a feature you could name: so networks are hard to explain

| Training | Predicting |
| - | - |
| <img src="https://upload.wikimedia.org/wikipedia/commons/0/0c/Simplified_neural_network_training_example.svg" alt="Training: connections between co-occurring features strengthen" style="width: 300px; display: block; margin: 0 auto;"> | <img src="https://upload.wikimedia.org/wikipedia/commons/b/b7/Simplified_neural_network_example.svg" alt="Predicting: an image's features point to its class" style="width: 300px; display: block; margin: 0 auto;"> |

<small>Images: Mikael Häggström, <a href="http://creativecommons.org/publicdomain/zero/1.0/deed.en">CC0</a>, after Ferrie and Kaiser, <em>Neural Networks for Babies</em> (2019)</small>

## Convolutional neural networks (CNN)
*   A **convolutional layer** slides small filters across the image. Each filter learns to respond to one simple pattern, such as an edge, wherever it appears
*   Layers of filters build on each other: edges, then shapes, then objects
*   The filters are learned in training, not designed by hand

![A typical CNN](https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png)

By Aphex34, [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0), via [Wikimedia Commons](https://commons.wikimedia.org/w/index.php?curid=45679374)

## Sequences: recurrent networks and transformers
*   The networks so far are **feed-forward**: each prediction starts from scratch, with no memory of earlier inputs
*   **Recurrent** networks (RNNs, and LSTMs among them) carry a memory from one input to the next. They were the standard for monitoring signals and text
*   **Transformers** take the whole sequence at once. **Self-attention** learns which earlier parts matter for each later one, such as a surgical event and a later infection. They are the networks behind large language models

![A recurrent neural network](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_RNN.svg)

## The trade-off
| What deep networks give | What they cost |
| :-- | :-- |
| They learn their own features from images, signals and text | Large datasets |
| High performance on complex problems | Computing time and money |
| A network trained on one task can be reused for a related one | Many hyperparameters to set |
| | Overfitting, with so many weights |
| | Hard to explain |

## Where depth pays
On a table like ours, a neural network rarely does much better than logistic regression. A comparison across healthcare studies found the same ([Issitt et al., 2022](https://doi.org/10.7759/cureus.22443)). **Deep** networks, with many hidden layers, pay where the features cannot be written down as columns:

*   **Images:** a chest X-ray is millions of pixels. **Convolutional** networks learn their own features from them: edges first, then shapes
*   **Signals and text:** **transformers**, the networks behind large language models, read a whole sequence at once

**» Investigate** — For each, **logistic regression or a deep network?**

1. In-hospital death from the first day's vital signs
2. A pneumothorax on a chest X-ray
3. Sepsis from the free text of nursing notes
4. Length of stay from age and admission type

# 6. Independent practice: a neural network for in-hospital death
From here to the **Regroup** is yours, in your pair: about 55 minutes.

* Each task is marked **» Modify** or **» Make**, and most ask for a **comment** recording a decision. Those comments are what we revisit in the Closing.
* **Hint** blocks fold open: try first.
* Every cell needs only the setup in section 1.

Work together, and call me over when you get stuck. **Understanding beats finishing.**

## Copy, then change
Copying working code is not cheating: it is how programmers work. The skill is in changing it.

1. **Find** the closest example above: the cell that does nearly what you need
2. **Copy** it into the practice cell
3. **List** what must change: the table, the columns, the model, the measure
4. **Change one thing, run it, and check** the output before the next change

When it breaks, **read the last line of the error first**:

| Error | Usually means |
| :-- | :-- |
| `NameError: name 'X_train' is not defined` | A cell above has not been run: **Runtime → Run before** |
| `KeyError: 'temprature_mean'` | A column name is misspelt |
| `TypeError: ... unexpected keyword argument` | A setting's name is misspelt |

## Live demo: copy, then change
Last week's cross-validation cell, copied in. I run it as it is, then change it into a network of ten hidden neurons, reading each error's last line as it comes.

**Before I change it:** which line of this cell has to change?

In [ ]:
# Last week, Session 5 Part 6: logistic regression, cross-validated on the training rows
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidate = LogisticRegression()

pipeline_candidate = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), candidate)
scores = cross_val_score(
    pipeline_candidate,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Mean AUC', round(scores.mean(), 3), ' spread between folds', round(scores.std(), 3))

## Part 1: Size, penalty and luck
**» Modify** — This cell cross-validates a network of ten hidden neurons on the training rows. It gets the raw `df_train[features]`, not `X_train`: the pipeline imputes and scales inside each fold. **Change `hidden_layer_sizes` to `(5,)` and then `(50,)`; then, back at `(10,)`, change `alpha` to 0.01 and then 1.** In the comment: **which change hurts, which helps, and which beat the spread between folds?**

In [ ]:
# What hurts, what helps, and what beats the spread between folds:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

network = MLPClassifier(hidden_layer_sizes=(10,), alpha=0.0001, random_state=0, max_iter=1000)
pipeline_net = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), network)
scores = cross_val_score(
    pipeline_net,
    df_train[features],
    df_train['died'],
    cv=GroupKFold(n_splits=5),
    groups=df_train['subject_id'],
    scoring='roc_auc',
)
print('Mean AUC', round(scores.mean(), 3), ' spread between folds', round(scores.std(), 3))

<details><summary><b>What you should find</b></summary>

50 neurons with almost no penalty does worst, and varies most between folds: it is overfitting, as in section 4. A strong penalty, `alpha = 1`, does best, but by less than the spread between folds, as do the other changes. Most of them are smaller than the spread between folds: Part 3 shows how to tell whether a difference like this is real.

</details>

**» Modify** — The same network, fitted on the training rows and tested once. Only its random starting weights change. **Change `random_state` to 1, 2, 3 and 4.** In the comment: how far does the test AUC move, and how does that compare with the gaps between the group's models?

In [ ]:
# How far the test AUC moves, against the gaps between the group's models:
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier

model_seed = MLPClassifier(hidden_layer_sizes=(10,), random_state=0, max_iter=1000)
model_seed.fit(X_train, y_train)

prob_seed = model_seed.predict_proba(X_test)[:, 1]
print('random_state', model_seed.random_state, ' test AUC', round(roc_auc_score(y_test, prob_seed), 3))

<details><summary><b>What you should find</b></summary>

The test AUC moves between about 0.79 and 0.81 with nothing changed but the starting weights: as much as the gap between some of the groups' models in section 2. One run of a network is one draw.

</details>

## Part 2: Tune it, then test it once
Part 1 was a grid search by hand: five settings, one run at a time, and an edit for each. `GridSearchCV` does every combination for you, by the same cross-validation, and keeps the best.

**» Make** — As your trio did, but for a network: **fill in the four blanks.** In the comment: the chosen values, both AUCs, and would you choose it over logistic regression?

<details><summary><b>Hint</b></summary>

The recap slide in section 2 has the search. Three sizes and three penalties make nine networks per fold, about a minute. Best value at the edge of the grid? Extend it that way. The test rows are `df_test[features]`.

</details>

In [ ]:
# Chosen values, CV AUC, test AUC, and would you choose it over logistic regression:
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline_net = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), MLPClassifier(random_state=0, max_iter=1000))

# 1. The grid: in a pipeline, each setting's name starts with its step's
grid = {
    'mlpclassifier__hidden_layer_sizes': [(5,), (10,), ____],
    'mlpclassifier__alpha': [0.0001, ____, 1],
}

# 2. The search, by cross-validation on the training rows, one patient per fold
search = GridSearchCV(pipeline_net, grid, cv=GroupKFold(n_splits=5), scoring='roc_auc')
search.fit(df_train[features], df_train['died'], groups=____)
print('Chosen:', search.best_params_, ' CV AUC:', round(search.best_score_, 3))

# 3. Once, at the end: the chosen network on the test rows
prob_search = search.predict_proba(____)[:, 1]
print('Test AUC:', round(roc_auc_score(df_test['died'], prob_search), 3))

## Part 3: Can logistic regression catch up?
**» Investigate** — The cell cross-validates logistic regression and the network Part 2's search chooses, on the same five folds, and prints every fold. **Does the network win in every fold? Is a lead smaller than the spread between folds still real?**

In [ ]:
# Does the network win in every fold:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def fold_aucs(model, table, columns):
    """The AUC on each of the same five folds of the training patients."""
    pipeline = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), model)
    return cross_val_score(pipeline, table[columns], table['died'], cv=GroupKFold(n_splits=5),
                           groups=table['subject_id'], scoring='roc_auc')


aucs_lr = fold_aucs(LogisticRegression(), df_train, features)
aucs_net = fold_aucs(MLPClassifier(hidden_layer_sizes=(50,), alpha=1, random_state=0, max_iter=1000), df_train, features)

print('Logistic regression:', aucs_lr.round(3), ' mean', round(aucs_lr.mean(), 3))
print('Network:            ', aucs_net.round(3), ' mean', round(aucs_net.mean(), 3))

**» Make** — Session 4's feature engineering, against the network. A patient far from a normal value, in either direction, is at risk: the cell adds the distance from 37 °C for temperature. **Add the same for the other four vital signs in the comments**, then compare with the network. In the comment: how much of the gap did you close, and why might you still choose logistic regression?

<details><summary><b>Hint</b></summary>

Copy the temperature line, and change the column, the normal value and the new column's name. Add each new name to `features_plus`. Run the cell above first: this one uses its `fold_aucs` and `aucs_net`.

</details>

In [ ]:
# How much of the gap you closed, and why you might still choose logistic regression:
table = df_train.copy()

# Distance from a normal value, in either direction
table['temperature_from_normal'] = (table['temperature_mean'] - 37).abs()
# heart_rate_mean: normal 80

# sbp_min: normal 110

# resp_rate_mean: normal 16

# glucose_max: normal 120

features_plus = features + ['temperature_from_normal']

aucs_plus = fold_aucs(LogisticRegression(), table, features_plus)
print('With the distances:', aucs_plus.round(3), ' mean', round(aucs_plus.mean(), 3))
print('Network:           ', aucs_net.round(3), ' mean', round(aucs_net.mean(), 3))

**» Modify (stretch)** — **Give the network the same features.** In the cell above, change `LogisticRegression()` to the network, `MLPClassifier(hidden_layer_sizes=(50,), alpha=1, random_state=0, max_iter=1000)`. Does knowing the shapes help the network too?

**» Modify (stretch)** — **Stop early.** `early_stopping=True` holds back a tenth of the training rows, and stops when the network's accuracy on them stops improving. **Change `early_stopping` to `True`.** In the comment: how many passes did it take, and what happened to the test AUC? Why might stopping early help?

In [ ]:
# Passes, test AUC, and why stopping early might help:
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier

model_early = MLPClassifier(early_stopping=False, random_state=0, max_iter=1000)
model_early.fit(X_train, y_train)

prob_early = model_early.predict_proba(X_test)[:, 1]
print('Passes:', model_early.n_iter_, ' test AUC', round(roc_auc_score(y_test, prob_early), 3))

**» Make (stretch)** — **A network for length of stay.** `MLPRegressor`, from the same module, is the network for regression. **Fit one to `los`, as Session 5's Part 1 did with `LinearRegression`, and compare its RMSE on the test rows with the straight line's 4.74 days and the mean's 4.80.** In the comment: did the network find what the line could not?

<details><summary><b>Hint</b></summary>

`MLPRegressor(random_state=0, max_iter=1000)`, then `fit(X_train, df_train['los'])` and `predict(X_test)`. `root_mean_squared_error` is in `sklearn.metrics`. Try `hidden_layer_sizes=(10,)` too.

</details>

In [ ]:
# Did the network find what the line could not:

# Your code here

## Regroup

Stop where you are: we come back together for the Closing.

If you didn't finish, finish after the session: Parts 1 and 2 matter most, and the stretch tasks are optional.

Finished early? Help a neighbour.

# 7. Closing
1. Your tuned network against logistic regression, on the same patients: **would you deploy it instead?** What would change your mind?
2. Logistic regression gives a clinician 26 numbers to read, one per feature and a bias. The default network has 2,701. **After the explainable AI talk: what would you show a clinician who asks why the network flagged their patient?**

## Wrap up
*   Logistic regression is one neuron: weights, a bias and the sigmoid. A hidden layer adds bends, so a network can learn a risk that rises at both ends of temperature
*   A network learns by gradient descent on the log loss, from random starting weights: the seed is another draw, like the split
*   More weights fit the training rows better, not the next patient. Regularise with `alpha`, and tune by cross-validation on the training rows, as your trio did
*   On our table of 25 columns, a network barely beats logistic regression and is harder to explain. Depth pays for images, signals and text
*   MIMIC data stay in your notebooks: do not share outputs, and do not paste rows into AI assistants

## Before the next session
1. **Finish the practice**, Parts 1 and 2, and check your decisions against the solutions notebook once it is released.
2. **Two talks open next week:** causal inference from observational data, and software as a medical device, from the groups listed on [Canvas](https://canvas.imperial.ac.uk/courses/8306). Everyone else: while they talk, write down one question you would ask.
3. **Optional:** deep learning on images. A guided notebook, about 45 minutes and nothing to submit, in which a convolutional network learns to tell organs apart on CT scans: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/exercises/Optional06_Deep_Learning_on_Images.ipynb).
4. Reading: Google's interactive introduction to neural networks, [developers.google.com/machine-learning/crash-course/neural-networks](https://developers.google.com/machine-learning/crash-course/neural-networks). Optional: a tutorial on convolutional networks for images, [www.tensorflow.org/tutorials/images/cnn](https://www.tensorflow.org/tutorials/images/cnn), and how far deep networks go, [The Illustrated AlphaFold](https://elanapearl.github.io/blog/2024/the-illustrated-alphafold/).

# Self-study appendices

## Bioinspiration
*   These models were created to mimic the function of biological neurons
*   By combining many neurons they are able to model complex relationships
*   In reality, the artificial neural networks are quite different from biological neural networks; however, they are very effective

## Complex neural network architecture
Networks with many neurons and many layers can model highly complex relationships

![Complex artificial neural network](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Complex-ANN.svg)

## The layers of a network
| **Aspect**                 | **Input Layer**                                    | **Hidden Layer(s)**                               | **Output Layer**                                  |
|----------------------------|----------------------------------------------------|---------------------------------------------------|---------------------------------------------------|
| **Role**                   | Receives and passes input data to the network      | Processes data, learns features and patterns      | Generates the final prediction or output          |
| **Data Handling**          | Takes raw input data (e.g., features, images)      | Transforms data through weighted connections      | Produces output suitable for the task (class/probability/value) |
| **Activation Function**    | None                                               | Non-linear functions (e.g., ReLU, Tanh, Sigmoid)  | Depends on the problem: Softmax, Sigmoid, Linear  |
| **Number of Neurons**      | Equal to the number of input features              | Varies based on network architecture              | Depends on the output type (e.g., 1 for regression, n for n-class classification) |
| **Complexity**             | Simple; no learning occurs in this layer           | Complex; learns hierarchical features and patterns| Simple; maps learned features to output space     |
| **Learning Role**          | No learning or transformations                     | Learns and extracts meaningful features           | Outputs the final result of the network’s learning|
| **Example**                | Each neuron represents a patient feature (e.g., age, heart rate, blood pressure) | Neurons learn interactions between features (e.g., BMI) | Outputs probability of mortality (e.g., risk score or binary prediction) |

## Softmax: more than two classes
*   There is a challenge with multi-classification problems where you want to have a probability value per class ...
*   Consider a classifier with three possible outputs (high, medium, and low risk) and your output neurons produce `[2.2, 0.4, 0.1]`, but you want to know the probability of being in each risk category
*   Softmax is a function that allows you to combine your output layer values together and produce a probability distribution across the classes such that all sum to 1 (i.e. probabilities)
    *   e.g. `[2.2, 0.4, 0.1]` becomes `[0.78, 0.13, 0.10]`
*   This function is computationally efficient during training (which we will consider next)

## Network initialisation
*   To start training a neural network on data, you need an initial 'guess' at the weights/biases
*   You can start by assigning the values randomly; however, the distribution should be carefully considered to avoid training instability (see later on)
*   Methods such as Xavier (Glorot) or He (Kaiming) initialisation define distributions to sample from for different activation functions

![Complex artificial neural network](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Training-ANN-2.svg)

## Forward propagation
*   The next step is to simply use your (semi-randomly initialised) network to make a prediction for your training data
*   For example:
    *   we have a `2.1` and `1.0` as the two features of the first sample in our training data
    *   the neural network output predicts a probability of `0.8`
    *   however, the real value should be False (`0.0`)
    *   the model is currently poorly trained

![Artificial neural network training 3](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Training-ANN-3.svg)

## Loss functions
*   To train the model, we need a 'loss function'
*   Simplistically, this is a measure of how 'bad' the model is at predicting
    *   i.e. a loss function of 0 would be correct
*   **Regression:** typically use the 'Mean Squared Error' (MSE)
*   **Classification:** typically uses the 'Cross-Entropy Loss'
*   There are others and you should explore these when you build a model - justify your choice

![Artificial neural network training 4](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Training-ANN-4.svg)

## Backpropagation: conceptually
*   The idea is to 'nudge' the weights in the network so they slightly reduce the loss at each iteration
*   We step backwards through the neural network, at each layer:
    *   Adjust the weights and bias so that the output moves in the direction that minimises the loss
    *   Adjust the weights such that larger changes are applied to greater improvements
    *   Propagate the required changes (errors) in the previous layer's outputs
*   Once we reach the input layer, the neural network should be slightly better at the trained sample
*   Repeating this process over and over causes the network's parameters to converge on a trained configuration

![Artificial neural network training 4](https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/Illustrations-SDSAI-05_Training-ANN-4.svg)

## Training challenges
There are some challenges in training neural networks ...

*   **Vanishing Gradients:** learning can become very slow or stop in early layers due to the very shallow gradients of the sigmoid and tanh activation functions
*   **Exploding Gradients:** learning becomes unstable due to large gradients resulting from poor initialisation
*   **Dead ReLU Units:** learning stops because ReLU neurons get stuck in the 0 region with no gradient to recover

## Deep neural network examples
| Model | What it is | Size | Trained on |
| :-- | :-- | :-- | :-- |
| **DenseNet-264** (2017) | Convolutional network for photographs | 264 layers, 33 million parameters | 1.3 million labelled photographs (ImageNet) |
| **RETFound** (2023) | Foundation model for retinal images | A 24-layer vision transformer | 1.6 million fundus photographs and OCT scans, Moorfields; two weeks on 8 GPUs |
| **Prov-GigaPath** (2024) | Foundation model for digital pathology | Two transformers: image tiles, then whole slides | 1.4 billion tiles from 171,189 slides of over 30,000 patients |
| **AlphaFold 3** (2024) | Predicts the 3D structure of proteins and their partners | 48 blocks, then a diffusion module | Structures in the Protein Data Bank |
| **DeepSeek-V3** (2024) | Language model, mixture of experts | 671 billion parameters, 37 billion used per token | 14.8 trillion tokens; 2.8 million GPU-hours |
| **GPT-4** (2023), *rumoured* | Language model, reportedly a mixture of experts | *Reported* 1.8 trillion parameters, 120 layers | *Reported* 13 trillion tokens; about $63 million |

<small>OpenAI, Google and Anthropic do not publish the size or training cost of their models: the GPT-4 figures come from a 2023 industry report and were never confirmed. DeepSeek publishes its own. A **mixture of experts** runs only part of the network for each token. Sources: [DenseNet](https://arxiv.org/abs/1707.06990), [RETFound](https://doi.org/10.1038/s41586-023-06555-x), [Prov-GigaPath](https://doi.org/10.1038/s41586-024-07441-w), [AlphaFold 3](https://doi.org/10.1038/s41586-024-07487-w), [DeepSeek-V3](https://arxiv.org/abs/2412.19437).</small>

## Neural networks against logistic regression
*   You now have lots of tools to address supervised machine learning problems; however, experience and reading up on the nuances of each method is the difference between using them well
*   REMEMBER... if your model does not have the key features of the system/disease/procedure/etc. you are trying to predict, no matter how complex you make your model or how many samples you collect, it will never improve its performance

![](https://assets.cureus.com/uploads/figure/file/292667/article_river_6a398820513311ecb45235ba7f0fde7e-LRvsNN_plot4.png)

Issitt R W, Cortina-Borja M, Bryant W, et al. (February 21, 2022) Classification Performance of Neural Networks Versus Logistic Regression Models: Evidence From Healthcare Practice. Cureus 14(2): e22443. doi:10.7759/cureus.22443

## Hyperparameter tuning
*   As models become more powerful, they tend to have increasing numbers of parameters
*   For example:
    *   Number of neighbours in KNN
    *   Depth of a decision tree
    *   Layers, neurons, activation functions, etc for neural networks
*   We call these **hyperparameters**
*   You have seen that variation in these parameters can effect model performance enormously

**How do you think we can ensure we have the best configuration for our model's hyperparameters?**

## Grid search
Involves systematically searching combinations of hyperparameters to find the best score

*   Approach guarantees a 'reasonable' proximity to the optimal values
*   `from sklearn.model_selection import GridSearchCV`

![Grid search hyperparameter optimisation](https://upload.wikimedia.org/wikipedia/commons/b/b6/Hyperparameter_Optimization_using_Grid_Search.svg)

By <a href="//commons.wikimedia.org/w/index.php?title=User:Alexander_Elvers&amp;action=edit&amp;redlink=1" class="new" title="User:Alexander Elvers (page does not exist)">Alexander Elvers</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=84255403">Link</a>

## Random search
Involves randomly searching combinations of hyperparameters to find the best score

*   Explores the continuous value space more efficiently than the grid search
*   `from sklearn.model_selection import RandomizedSearchCV`

![Random search hyperparameter optimisation](https://upload.wikimedia.org/wikipedia/commons/7/74/Hyperparameter_Optimization_using_Random_Search.svg)

By <a href="//commons.wikimedia.org/w/index.php?title=User:Alexander_Elvers&amp;action=edit&amp;redlink=1" class="new" title="User:Alexander Elvers (page does not exist)">Alexander Elvers</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=84255404">Link</a>

## Bayesian optimisation
Involves using an optimisation approach to search in areas where it expects the performance to be better

*   Typically finds optima more quickly than random search
*   `from skopt import BayesSearchCV`

![Bayesian Optimisation hyperparameter optimisation](https://upload.wikimedia.org/wikipedia/commons/3/3c/Hyperparameter_Optimization_using_Tree-Structured_Parzen_Estimators.svg)

By <a href="//commons.wikimedia.org/w/index.php?title=User:Alexander_Elvers&amp;action=edit&amp;redlink=1" class="new" title="User:Alexander Elvers (page does not exist)">Alexander Elvers</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="https://creativecommons.org/licenses/by-sa/4.0" title="Creative Commons Attribution-Share Alike 4.0">CC BY-SA 4.0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=84255405">Link</a>

## Model sensitivity and stability
*   You can use the hyperparameter optimisation to assess the sensitivity and stability of a model
*   If small changes in parameters lead to large changes in performance, the model is 'sensitive' and maybe overfit
*   If large changes in parameters lead to small changes in performance, the model is either already at its limit, underfit, or suffering from some other issues

## Exercise: an artificial neuron by hand
For a three input neuron with the following input values and parameters, can you calculate (with simple Python) the output value of the neuron for the input `A Values` and the `B Values`?

You should get a separate output value for each input set

### Parameters
| Parameter | Value |
| --------- | ----- |
| weight 1  | 0.5   |
| weight 2  | 1.7   |
| weight 3  | 0.001 |
| bias      | -52   |
| Activation function | ReLU |

### Inputs
| Input     | A Values  | B Values  |
| --------- | --------- | --------- |
| input 1   | 65 kg     | 82 kg     |
| input 2   | 1.7 m     | 1.85 m    |
| input 3   | 8200 c/uL | 9800 c/uL |

In [ ]:
w1 = 0.5
w2 = 1.7
w3 = 0.001
bias = -52

def relu(x):
    pass # Write your ReLU function here

va1 = 65
va2 = 1.7
va3 = 8200

vb1 = 82
vb2 = 1.85
vb3 = 9800

# Calculate neuron outputs
# output1 = ?
# output2 = ?